# Person 4 — Join Pipeline (fast path)

**Do this once:**
1. Upload this notebook to Google Colab (or open from Drive).
2. Runtime → **Run all**.
3. Approve Drive access when prompted.

It finds your V2 files on Drive, subsets to the Person-2 text-feature IDs, and writes:
- `MKT4220_Project/data/joined_v2/04_review_features.jsonl`
- `MKT4220_Project/data/joined_v2/04_review_features.csv`
- `MKT4220_Project/data/joined_v2/acceptance_report.json`

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import json
import csv
from collections import defaultdict, Counter
from statistics import mean
from datetime import datetime, timezone

PROJECT = Path('/content/drive/MyDrive/MKT4220_Project')
assert PROJECT.exists(), f'Missing project folder: {PROJECT}'

OUT = PROJECT / 'data' / 'joined_v2'
OUT.mkdir(parents=True, exist_ok=True)

def find_file(filename: str) -> Path:
    matches = sorted(PROJECT.rglob(filename))
    if not matches:
        raise FileNotFoundError(
            f'Could not find {filename} under {PROJECT}. '
            'Upload/copy it into the project Drive folder and re-run.'
        )
    # Prefer paths under data/ when multiple copies exist
    preferred = [p for p in matches if '/data/' in str(p).replace('\\', '/')]
    chosen = preferred[0] if preferred else matches[0]
    print(f'{filename} -> {chosen}')
    return chosen

REVIEWS_SRC = find_file('reviews_image_filtered_v2.jsonl')
TEXT_SRC = find_file('text_features_image_v2_test.jsonl')
IMAGES_SRC = find_file('image_features_available_v2.jsonl')
print('Output folder:', OUT)

## 1. Load text-feature IDs

In [ ]:
def load_jsonl(path):
    rows = []
    with path.open('r', encoding='utf-8') as f:
        for line in f:
            if line.strip():
                rows.append(json.loads(line))
    return rows

text_rows = load_jsonl(TEXT_SRC)
target_ids = {r['review_id'] for r in text_rows}
print('text rows:', len(text_rows))
print('unique review_ids:', len(target_ids))
print('text run_ids:', sorted({r.get('run_id') for r in text_rows}))
assert len(text_rows) == len(target_ids), 'duplicate text review_ids'

## 2. Subset reviews + image features

In [ ]:
reviews = []
with REVIEWS_SRC.open('r', encoding='utf-8') as f:
    for line in f:
        if not line.strip():
            continue
        r = json.loads(line)
        if r['review_id'] in target_ids:
            reviews.append(r)

images = []
with IMAGES_SRC.open('r', encoding='utf-8') as f:
    for line in f:
        if not line.strip():
            continue
        r = json.loads(line)
        if r['review_id'] in target_ids:
            # Normalize optional fields for contract join
            if 'image_source' not in r:
                r['image_source'] = 'customer_review'
            images.append(r)

review_ids = {r['review_id'] for r in reviews}
missing = sorted(target_ids - review_ids)
print('matched reviews:', len(reviews))
print('matched image rows:', len(images))
print('missing reviews for text ids:', len(missing))
if missing[:5]:
    print('missing examples:', missing[:5])
assert not missing, 'Every text review_id must exist in reviews file'

# Keep review order stable: follow text file order when possible
review_by_id = {r['review_id']: r for r in reviews}
reviews = [review_by_id[t['review_id']] for t in text_rows]

## 3. Join into 04_review_features

In [ ]:
text_by_id = {t['review_id']: t for t in text_rows}
images_by_review = defaultdict(list)
for img in images:
    images_by_review[img['review_id']].append(img)

batch_image_run_ids = {img.get('run_id') for img in images if img.get('run_id')}
batch_image_run_id = next(iter(batch_image_run_ids)) if len(batch_image_run_ids) == 1 else None

joined = []
status_counts = Counter()

for review in reviews:
    rid = review['review_id']
    text = text_by_id[rid]
    imgs = images_by_review.get(rid, [])

    ok_scores = [
        img['text_image_cosine']
        for img in imgs
        if img.get('status') == 'ok' and img.get('text_image_cosine') is not None
    ]
    usable = len(ok_scores)
    total = len(imgs) if imgs else int(review.get('image_count') or 0)

    if total == 0:
        image_status = 'no_images'
    elif usable == 0:
        image_status = 'failed'
    elif usable < total:
        image_status = 'partial'
    else:
        image_status = 'ok'
    status_counts[image_status] += 1

    run_ids = {img.get('run_id') for img in imgs if img.get('run_id')}
    if len(run_ids) == 1:
        image_run_id = next(iter(run_ids))
    elif imgs:
        image_run_id = sorted(imgs, key=lambda x: x.get('image_index', 0))[0].get('run_id')
    else:
        image_run_id = batch_image_run_id

    joined.append({
        'review_id': rid,
        'parent_asin': review['parent_asin'],
        'split': review['split'],
        'rating': review['rating'],
        'helpful_vote': review['helpful_vote'],
        'verified_purchase': review.get('verified_purchase'),
        'timestamp_ms': review['timestamp_ms'],
        'image_count': review['image_count'],
        'text_word_count': text.get('text_word_count'),
        'sentiment_negative': text.get('sentiment_negative'),
        'sentiment_neutral': text.get('sentiment_neutral'),
        'sentiment_positive': text.get('sentiment_positive'),
        'usable_image_count': usable,
        'image_status': image_status,
        'text_image_cosine_mean': mean(ok_scores) if ok_scores else None,
        'text_image_cosine_min': min(ok_scores) if ok_scores else None,
        'text_run_id': text.get('run_id'),
        'image_run_id': image_run_id,
    })

print('joined rows:', len(joined))
print('image_status:', dict(status_counts))
assert len(joined) == len(reviews) == len(text_rows)

## 4. Save outputs

In [ ]:
fields = list(joined[0].keys())
jsonl_path = OUT / '04_review_features.jsonl'
csv_path = OUT / '04_review_features.csv'
report_path = OUT / 'acceptance_report.json'

with jsonl_path.open('w', encoding='utf-8') as f:
    for row in joined:
        f.write(json.dumps(row) + '\n')

with csv_path.open('w', encoding='utf-8', newline='') as f:
    writer = csv.DictWriter(f, fieldnames=fields)
    writer.writeheader()
    for row in joined:
        out = {k: ('' if v is None else v) for k, v in row.items()}
        writer.writerow(out)

report = {
    'passed': True,
    'generated_at_utc': datetime.now(timezone.utc).isoformat(),
    'counts': {
        'reviews': len(reviews),
        'text_features': len(text_rows),
        'image_features': len(images),
        'joined_rows': len(joined),
        'image_status': dict(status_counts),
    },
    'sources': {
        'reviews': str(REVIEWS_SRC),
        'text': str(TEXT_SRC),
        'images': str(IMAGES_SRC),
    },
}
report_path.write_text(json.dumps(report, indent=2), encoding='utf-8')

print('DONE')
print('Saved:', jsonl_path)
print('Saved:', csv_path)
print('Saved:', report_path)
print('Preview:')
for row in joined[:3]:
    print({
        k: row[k]
        for k in [
            'review_id', 'split', 'image_status',
            'usable_image_count', 'text_image_cosine_mean', 'sentiment_positive'
        ]
    })